<a href="https://colab.research.google.com/github/Swas26/NLP-ESG/blob/main/data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing
**Pipeline:** ESG PDF reports + NewsAPI articles → cleaned, chunked, combined CSV

**Only needs to be re-run when source data changes.**
The filtering/scoring notebook pulls outputs directly from GitHub.

**Outputs pushed to `data/processed/` on GitHub:**
- `esg_chunks.csv` — chunked text from 12 company ESG PDFs
- `news_articles.csv` — news articles fetched from NewsAPI
- `combined.csv` — merged dataset, input for filtering/scoring notebook

**Before running:**
1. Click the 🔑 icon in the left sidebar
2. Add a secret called `GITHUB_TOKEN` with your personal access token
   - Get one at: github.com/settings/tokens → Generate new token (classic) → tick `repo`
3. Toggle **Notebook access** on

In [ ]:
# --- SETUP ---
# Installs deps, loads GitHub token from Colab Secrets, downloads shared ESG PDFs

!pip install pdfplumber newsapi-python gdown -q

import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

FOLDER_ID  = '1SUewYSR3PYYAMUHi9Egqc3Bd8wA0zKeY'
DATA_DIR   = '/content/data'
OUTPUT_DIR = '/content/outputs'
REPO       = 'Swas26/NLP-ESG'
REPO_DIR   = '/content/repo'
GIT_EMAIL  = 'aatman.dwivedi@student.uts.edu.au'
GIT_NAME   = 'Swas26'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Download ESG PDFs from shared Google Drive folder (public link, no sign-in needed)
if not os.path.exists(DATA_DIR):
    print('Downloading ESG PDFs...')
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O {DATA_DIR}
else:
    print('Data already present, skipping download.')

print('Setup complete ✓')

In [ ]:
# --- PART 1: ESG PDF CHUNKING ---
# Reads each company PDF, extracts raw text, splits into ~400-word chunks
# Chunking is necessary because ClimateBERT has a max token limit per input

import pdfplumber
import pandas as pd

filename_to_company = {
    'Woolworths Australia': 'Woolworths Australia',
    'UniQlo': 'UniQlo',
    'Santos.pdf': 'Santos',
    'Qantas': 'Qantas',
    'NBN.pdf': 'NBN',
    'Muji.pdf': 'Muji',
    'LVMH.pdf': 'LVMH',
    'Gucci.pdf': 'Gucci',
    'Coles Group.pdf': 'Coles Group',
    'CBA.pdf': 'CBA',
    'ANZ': 'ANZ',
    'AGL.pdf': 'AGL',
}

all_chunks = []
CHUNK_SIZE = 400

for filename, company_name in filename_to_company.items():
    filepath = os.path.join(DATA_DIR, filename)

    if not os.path.exists(filepath):
        print(f'MISSING: {filename} — skipping')
        continue

    full_text = ''
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + ' '

    words = full_text.split()
    chunks_added = 0

    for i in range(0, len(words), CHUNK_SIZE):
        chunk = ' '.join(words[i:i+CHUNK_SIZE])
        if len(chunk.strip()) > 50:
            all_chunks.append({
                'company': company_name,
                'source_type': 'esg_report',
                'text': chunk
            })
            chunks_added += 1

    print(f'{company_name}: {len(words)} words → {chunks_added} chunks')

esg_df = pd.DataFrame(all_chunks)
esg_df.to_csv(f'{OUTPUT_DIR}/esg_chunks.csv', index=False)
print(f'\nDone — {len(esg_df)} total ESG chunks saved')

In [ ]:
# --- PART 2: NEWS ARTICLE SCRAPING ---
# Fetches sustainability-related news for each company via NewsAPI
# Free tier: 30 articles/company, 1 req/sec — don't re-run unnecessarily

from newsapi import NewsApiClient
import time

api = NewsApiClient(api_key='822e469b13934cf2adc43ce8872be994')

companies = [
    'AGL', 'CBA', 'ANZ', 'Qantas', 'Santos', 'NBN',
    'Muji', 'LVMH', 'UniQlo', 'Woolworths Australia', 'Coles Group', 'Gucci'
]

all_articles = []

for company in companies:
    response = api.get_everything(
        q=f'{company} sustainability OR environment OR ESG OR emissions',
        language='en',
        page_size=30,
        sort_by='relevancy'
    )

    for article in response['articles']:
        all_articles.append({
            'company': company,
            'source_type': 'news',
            'text': str(article['title']) + '. ' + str(article['description']),
            'published_at': article['publishedAt'],
            'source': article['source']['name']
        })

    print(f'{company}: {len(response["articles"])} articles fetched')
    time.sleep(1)

news_df = pd.DataFrame(all_articles)
news_df.dropna(subset=['text'], inplace=True)
news_df.to_csv(f'{OUTPUT_DIR}/news_articles.csv', index=False)
print(f'\nDone — {len(news_df)} articles saved')

In [ ]:
# --- PART 3: COMBINE ESG + NEWS ---
# Merges both sources into one dataset
# combined.csv is the single input for the filtering/scoring notebook

esg_df  = pd.read_csv(f'{OUTPUT_DIR}/esg_chunks.csv')
news_df = pd.read_csv(f'{OUTPUT_DIR}/news_articles.csv')

news_df = news_df[['company', 'source_type', 'text']]  # align columns

combined_df = pd.concat([esg_df, news_df], ignore_index=True)
combined_df.to_csv(f'{OUTPUT_DIR}/combined.csv', index=False)

print(f'ESG chunks:     {len(esg_df)}')
print(f'News articles:  {len(news_df)}')
print(f'Total combined: {len(combined_df)}')
print(combined_df['source_type'].value_counts())

In [ ]:
# --- PUSH OUTPUTS TO GITHUB ---
# Clones repo, copies CSVs in, pushes to data/processed/
# Teammates just run `git pull` to get latest files — no reprocessing needed

import shutil

!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git {REPO_DIR} -q

processed_dir = f'{REPO_DIR}/data/processed'
os.makedirs(processed_dir, exist_ok=True)

for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.csv'):
        shutil.copy(f'{OUTPUT_DIR}/{f}', f'{processed_dir}/{f}')
        print(f'Copied {f}')

!cd {REPO_DIR} && git config user.email "{GIT_EMAIL}"
!cd {REPO_DIR} && git config user.name "{GIT_NAME}"
!cd {REPO_DIR} && git add data/processed/
!cd {REPO_DIR} && git commit -m "update preprocessed data outputs"
!cd {REPO_DIR} && git push

print('\nAll outputs pushed to GitHub → data/processed/')